In [1]:
from qldpc_sim import *
from qldpc_sim.qldpc_experiment import *
from qldpc_sim.qec_code import *
from qldpc_sim.data_structure import *
import stim
import numpy as np

In [2]:
from qldpc_sim.qec_code.rotated_surface_code import RotatedSurfaceCode


def setup_cnot_exp():
    patch1 = RotatedSurfaceCode.from_distance(3, code_name="p1")
    patch2 = RotatedSurfaceCode.from_distance(3, code_name="p2")
    qm = qldpc_experiment.QuantumMemory(size=600)
    patches = [patch1, patch2]
    mapqb = {}
    for c in patches:
        for lq in c.logical_qubits:
            mapqb[lq.logical_x] = c
            mapqb[lq.logical_z] = c

    ctx = Context(
        codes=[patch1, patch2],
        logical_qubits=patch1.logical_qubits + patch2.logical_qubits,
        initial_assignement=mapqb,
        memory=qm,
    )

    return ctx

In [3]:
from qldpc_sim.ckbb_surgery.measurement import CKBBMeasurement
from qldpc_sim.rsc_surgery.rsc_surgery import SurgeMeasurement
from qldpc_sim.qldpc_experiment.qec_gadget import Readout

def get_joinm_program(initial_state1, initial_state2, joint_pauli, readout_basis):
    context = setup_cnot_exp()

    p1 = context.codes[0]
    p2 = context.codes[1]
    if joint_pauli == "XX":
        ckbbm = [
            CKBBMeasurement(
                distance=2,
                context=context,
                tag="CKBBM",
                logical_targets=[
                    p2.logical_qubits[0].logical_x,
                    p1.logical_qubits[0].logical_x,
                ],
            ),
        ]
    else:
        ckbbm = [
            CKBBMeasurement(
                distance=2,
                context=context,
                tag="CKBBM",
                logical_targets=[
                    p2.logical_qubits[0].logical_z,
                    p1.logical_qubits[0].logical_z,
                ],
            ),
        ]
    return (
        [
            InitializeCode(
                code=p1,
                context=context,
                tag=f"init_{p1.id}",
                initial_state=initial_state1,
            )
        ]
        + [
            InitializeCode(
                code=p2,
                context=context,
                tag=f"init_{p2.id}",
                initial_state=initial_state2,
            )
        ]
        + [
            StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
            for p in [p1, p2]
        ]
        + ckbbm
        + [
            StabMeasurement(code=p, context=context, tag=f"isb_3_{p.id}", round=3)
            for p in [p1, p2]
        ]
        + [
            LM(
                logical_targets=[p1.logical_qubits[0].logical_x if readout_basis == "X" else p1.logical_qubits[0].logical_z],
                context=context,
                tag=f"p1",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
                reset_qubits=False,
            ),
            LM(
                logical_targets=[p2.logical_qubits[0].logical_x if readout_basis == "X" else p2.logical_qubits[0].logical_z],
                context=context,
                tag=f"p2",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
                reset_qubits=False,
            ),
            Readout(
                code=p1,
                context=context,
                tag=f"readout_{p1.id}",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
            ),
            Readout(
                code=p2,
                context=context,
                tag=f"readout_{p2.id}",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
            ),
        ]
    ), context

In [4]:
from typing import Dict, List, Set, Tuple

from qldpc_sim.qldpc_experiment.qec_gadget import LogicGadget


def evaluate(ctx, program: List[QECGadget]) -> Tuple[List[str], Dict[str, Set[int]]]:
    """Evaluate a qLDPC program in the context.

    Args:
        program (List[QECGadget]): The list of gadgets to evaluate in the context.
    """
    logical_outcomes: Dict[str, Set[int]] = {}
    instructions = []
    for gadget in program:
        compilers, outcomes = gadget.build_compiler_instructions()
        for compiler in compilers:
            n_instructions, measurements = compiler.compile(ctx.memory)
            instructions.extend(n_instructions)
            if measurements is not None:
                ctx.record.add_measurements(measurements)
        for outcome in outcomes:
            ctx.record.add_event(outcome)
            if outcome.type == EventType.FRAME_CORRECTION:
                logical_qubit = ctx.map_operator_to_qubits[outcome.target]
                ctx.frame_tracker.add_correction(
                    logical_qubit, outcome.target.logical_type, outcome.measured_nodes
                )
            if outcome.type == EventType.OBSERVABLE:
                if isinstance(outcome.target, LogicalOperator):
                    if outcome.tag not in logical_outcomes:
                        logical_outcomes[outcome.tag] = set()
                    logical_outcomes[outcome.tag] |= set(outcome.measured_nodes)
                    logical_outcomes[outcome.tag] |= set(
                        ctx.frame_tracker.get_correction(
                            ctx.map_operator_to_qubits[outcome.target],
                            outcome.target.logical_type,
                        )
                    )

    return instructions, logical_outcomes, ctx.record

In [5]:
from collections import Counter
from itertools import combinations
import json
from pathlib import Path

from qldpc_sim.qldpc_experiment import context
from qldpc_sim.qldpc_experiment.interpreter import concat_events_per_sample, run, xor_event_nodes

prog, ctx = get_joinm_program(
    initial_state1=PauliEigenState.X_plus,
    initial_state2=PauliEigenState.X_plus,
    joint_pauli="ZZ",
    readout_basis="X",
)

p, lo, rec = evaluate(ctx, prog)

print(f"recorded events: {[(e.tag, v) for e, v in rec.events.items()]}")
print("Recorded measurements and outcomes:")
for n, m in rec.measurements.items():
    print(f"{n.tag} - {m}")

# Build event-to-measurement mapping once; some derived events may not map cleanly.
event_idx_map = None
try:
    event_idx_map = rec.event_to_measurement_idx
except ValueError as err:
    print(f"Warning: could not build event_to_measurement_idx ({err})")

for event, value in rec.events.items():
    print(event)
    print(f"{event.tag}: {[e.tag for e in event.measured_nodes]}")
    included_idx = event_idx_map.get(event) if event_idx_map is not None else None
    print(f"Included index : {included_idx}")

print("Logical outcomes:")
for key, value in lo.items():
    print(f"  {key}: {[v.tag for v in value]}")

recorded events: [('StabMeasurement_isb_2_680829d0-bb6e-46d8-92fa-0c5d997fa279', 40), ('StabMeasurement_isb_2_2a5c34da-ecad-4825-b5c9-e3e3344e6f62', 64), ('CKBBM_parity_outcome', 146), ('CKBBMlog_corr_3b16859e-1120-497f-be68-2e8559cba582', 146), ('StabMeasurement_isb_3_680829d0-bb6e-46d8-92fa-0c5d997fa279', 170), ('StabMeasurement_isb_3_2a5c34da-ecad-4825-b5c9-e3e3344e6f62', 194), ('LogicalMeasurement_p1', 197), ('LogicalMeasurement_p2', 200), ('Readout_readout_680829d0-bb6e-46d8-92fa-0c5d997fa279', 217), ('Readout_readout_2a5c34da-ecad-4825-b5c9-e3e3344e6f62', 234)]
Recorded measurements and outcomes:
c_x_2_p1 - [0, 16, 24, 32, 93, 126, 146, 154, 162, 201]
c_z_0_p1 - [1, 17, 25, 33, 64, 97, 147, 155, 163, 205]
c_x_3_p1 - [2, 18, 26, 34, 78, 111, 148, 156, 164, 208]
c_x_1_p1 - [3, 19, 27, 35, 79, 112, 149, 157, 165, 210]
c_x_0_p1 - [4, 20, 28, 36, 82, 115, 150, 158, 166, 212]
c_z_2_p1 - [5, 21, 29, 37, 69, 102, 151, 159, 167, 213]
c_z_3_p1 - [6, 22, 30, 38, 83, 116, 152, 160, 168, 214]